In [ ]:
# imports
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display

import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..','..')))
from ai_tools.tools import LLMQuery, handle_tool_call
sys.path.append(os.path.abspath(os.path.join(os.getcwd())))
from pokemon import PokemonAPIClient, TOOLS 

In [2]:
pokemon_client = PokemonAPIClient()
functions = [getattr(pokemon_client, tool['function']['name']) for tool in TOOLS]

In [ ]:
client = LLMQuery(system_prompt=pokemon_client.get_system_prompt(), functions=functions,tools=TOOLS, model ="" )
client.

[]

In [39]:
def chat(user_input):
    response = client.query(user_input)
    response = client.get_tool_responses()
    return response

In [45]:
r = chat("""Professor, mein großer Bruder lacht mich aus! Er sagt, ich soll meinem Scherox unbedingt die Attacke 'Patronenhieb' beibringen. Aber ich hab im Pokédex geschaut und die hat nur Stärke 40! Das ist doch voll mickrig, oder?

Er meinte irgendwas von wegen, wenn mein Scherox die Fähigkeit 'Techniker' hat, wird das voll stark. Kannst du das mal für mich nachrechnen? Wie viel Stärke hat Patronenhieb am Ende wirklich, wenn man die Fähigkeit und den STAB-Bonus mitrechnet? Ist das stärker als ein normaler Eisenschädel (Stärke 80)?""" )


In [ ]:
# gradio chatbot with model selection
tts_client = LLMQuery(model="gpt-4o-mini-tts")

model_names = [
    "gpt-4o-mini", "gpt-5-nano", "gpt-5-mini", "gpt-5.1", "gpt-5.2", 
    "gpt-4.1-mini", "gpt-5.2-pro", 
    "llama3.2", "deepseek-r1:1.5b",
    "gemini-3-pro-preview", "gemini-2.5-flash", "gemini-2.5-flash-lite", 
    "gemini-flash-latest", "gemini-flash-lite-latest",
    "anthropic/claude-sonnet-4.5", "openai/gpt-oss-120b", "deepseek/deepseek-v3.2", "x-ai/grok-4"
]
def chat(message, history):
    # Query the LLM
    response = client.query(message)
    # Handle any tool calls that occurred
    response = client.get_tool_responses()

    # Generate TTS
    audio_bytes = tts_client.generate_tts(response, voice="onyx")
    return response, audio_bytes


def set_model(model_name):
    # Update the model in the LLMQuery client
    client.model = model_name


# Gradio Interface
with gr.Blocks() as app:
    gr.Markdown("# Pokémon Chatbot")

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=model_names,
            value=client.model,
            label="Select LLM Model",
            interactive=True,
        )
        audio_output = gr.Audio(label="Professor Eich's Voice", autoplay=True)

    # Using ChatInterface for the chat UI
    chat_interface = gr.ChatInterface(
        fn=chat, type="messages", additional_outputs=[audio_output]
    )

    model_dropdown.change(fn=set_model, inputs=model_dropdown, outputs=None)

app.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


: 

In [46]:
display(Markdown(r))

Kurz gesagt: Mit Techniker und STAB erreicht Patronenhieb (Bullet Punch) eine Stärke von 90. Eisenschädel (Iron Head) hat bei STAB 120. Also ist Patronenhieb nicht stärker als Eisenschädel — Eisenschädel ist rein rechnerisch stärker, wenn man nur die Rundstärke vergleicht.

Hier die Rechnung im Schnelldurchlauf:
- Move: Patronenhieb (Bullet Punch)
  - Basisstärke: 40
  - Techniker wirken: 40 × 1,5 = 60
  - STAB (Schaden durch gleichen Typ): 60 × 1,5 = 90
- Eisenschädel (Iron Head)
  - Basisstärke: 80
  - STAB: 80 × 1,5 = 120

Zusatzinfos:
- Scherox hat die Fähigkeit Techniker, und Bullet Punch ist ein Steel-M Move mit Basisstärke 40, damit gilt die Techniker-Regel.
- Bullet Punch hat zudem Priorität, was es in Kämpfen nützlich macht, aber die rohe Stärke bleibt 90. Eisenschädel bleibt mit 120 stärker, sofern sonst keine zusätzlichen Multiplikatoren (Dauern, Itemeffekte etc.) dazukommen.
- Die tatsächliche Schadenshöhe im Kampf hängt zusätzlich von Angriffswerten (hier Scheroxs Angriff 130), Verteidigung des Gegners, Typenkombinationen, Effektivität und Zufallsfaktoren ab.

Wenn du willst, sag mir, gegen welches Typen oder welches Pokémon du kämpfen willst — dann berechne ich dir die voraussichtliche Schadensspanne unter konkreten Bedingungen.

In [41]:
display(Markdown(r))

Hallo junger Trainer! Kurze Antwort zuerst:

- Mit der **Fähigkeit Techniker** wird das basisstarke **Patronenhieb** (engl. Bullet Punch) von 40 auf 60 Schaden-Power erhöht.
- Da **Scherox** (Bug/Flying) den Zugstick nicht mit dem Typ von Patronenhieb teilt, gibt es kein STAB-Bonus.
- Gegenüber dem normalen **Eisenschädel** (Iron Head) mit 80 BP bleibt Patronenhieb bei 60 BP also deutlich schwächer.

Detailliert erklärt:

- Patronenhieb (Bullet Punch)
  - Basis-Power: **40**
  - Typ: **Fighting**
  - Priorität: +1
  - STAB: Nein, weil Scherox Bug/Flying ist und Patronenhieb Fighting ist (kein Typen-Match)
  - Mit Techniker: 40 × 1,5 = **60** BP

- Eisenschädel (Iron Head)
  - Basis-Power: **80**
  - Typ: **Steel**
  - STAB: Nein (Scherox ist nicht Steel)
  - Ohne Bonus: **80** BP

Schlussfolgerung:
- Patronenhieb mit Techniker erreicht **60 BP**, Iron Head hat **80 BP**. Iron Head ist also stärker in der reinen Base-Power-Verrechnung.

Wenn du magst, rechne ich dir gerne die echte Schadenshöhe aus, sobald du mir Level, Angriffs-/Verteidigungswerte, EVs/IVs und ob du physisch oder speziell angreifst, gibst. Dann schauen wir uns die genaue Schadensformel an.